# Luggage Decision Data — Ingestion

Parses raw Pega decision JSON exports, extracts all luggage weight variant
(L5B15 / L5B20 / L5B25 / L5B30 / L5B40 / L5B50 / CLUG) scoring events for the
**Email / Outbound / Luggage / Sales** partition, and writes a clean Parquet file
to `data/processed/`.

**To add new data:** drop additional JSON export files into `data/decisions/` and re-run.
Deduplication on `modelExecutionID` ensures records are never double-counted.

In [1]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import polars as pl
from IPython.display import HTML, display
from tabulate import tabulate
from tqdm import tqdm

from my_project.parsing import extract_l5b15_rows, LUGGAGE_VARIANTS

print(f"Imports OK — targeting: {sorted(LUGGAGE_VARIANTS)}")

Imports OK — targeting: ['CLUG', 'L5B15', 'L5B20', 'L5B25', 'L5B30', 'L5B40', 'L5B50']


In [2]:
DECISIONS_DIR = Path("../data/decisions")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / "luggage_email_outbound.parquet"

# All decision JSON exports — add new files to data/decisions/ and re-run
JSON_FILES = sorted(DECISIONS_DIR.glob("*.json"))
print(f"Found {len(JSON_FILES)} source file(s):")
for f in JSON_FILES:
    print(f"  {f.name}")

Found 1 source file(s):
  data.json


## Load & parse raw records

In [3]:
all_rows = []

for filepath in JSON_FILES:
    print(f"\nProcessing {filepath.name}...")
    with open(filepath, encoding="utf-8") as f:
        total = sum(1 for _ in f)

    records = []
    with open(filepath, encoding="utf-8") as f:
        for line in tqdm(f, total=total, desc=filepath.name):
            records.append(json.loads(line))

    df_raw = pl.DataFrame(records)
    print(f"  {df_raw.shape[0]:,} raw interactions, {df_raw.shape[1]} columns")

    rows = extract_l5b15_rows(df_raw)   # extracts all LUGGAGE_VARIANTS by default
    print(f"  {len(rows):,} luggage scoring events extracted")
    all_rows.extend(rows)

print(f"\nTotal rows before deduplication: {len(all_rows):,}")


Processing data.json...


data.json: 100%|██████████| 16216/16216 [00:03<00:00, 4086.17it/s]


  16,216 raw interactions, 9 columns
  54,412 luggage scoring events extracted

Total rows before deduplication: 54,412


## Deduplicate & filter

In [4]:
df_all = pd.DataFrame(all_rows)

# Deduplicate on modelExecutionID (safe to re-run with overlapping exports)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["modelExecutionID"])
print(f"After dedup: {len(df_all):,} rows  (removed {before - len(df_all):,} duplicates)")

# Filter to the channel/partition of interest
df = df_all[
    (df_all["pyChannel"]   == "Email") &
    (df_all["pyDirection"] == "Outbound") &
    (df_all["pyGroup"]     == "Luggage") &
    (df_all["pyIssue"]     == "Sales")
].copy().reset_index(drop=True)

# Parse interaction timestamp
df["pxDecisionTime"] = pd.to_datetime(
    df["pxDecisionTime"], format="%Y%m%dT%H%M%S.%f %Z", utc=True, errors="coerce"
)

print(f"After filter (Email / Outbound / Luggage / Sales): {len(df):,} rows")
print(f"Columns: {df.shape[1]}")
df.head(3)

After dedup: 54,166 rows  (removed 246 duplicates)
After filter (Email / Outbound / Luggage / Sales): 6,164 rows
Columns: 442


,pxInteractionID,pxSubjectID,modelExecutionID,modelVersion,pxDecisionTime,pyName,pyChannel,pyDirection,pyGroup,pyIssue,...,IH.Email.Inbound.Clicked.pxLastOutcomeTime.DaysSince,IH.Email.Outbound.Rejected.pxLastOutcomeTime.DaysSince,IH.Email.Inbound.Rejected.pyHistoricalOutcomeCount,IH.Email.Outbound.Rejected.pyHistoricalOutcomeCount,IH.Email.Inbound.Rejected.pxLastGroupID,IH.Email.Outbound.Rejected.pxLastGroupID,IH.Email.Inbound.Rejected.pxLastOutcomeTime.DaysSince,IH.Push.Outbound.Clicked.pyHistoricalOutcomeCount,IH.Push.Outbound.Clicked.pxLastGroupID,IH.Push.Outbound.Clicked.pxLastOutcomeTime.DaysSince
0,7571315193780600815,M-15958320,4a389332-9591-4477-9280-00cc12098b7a,108d8fa1-b0ad-5211-9dae-98487aef9a4d,2026-04-19 10:28:50.166000+00:00,CLUG,Email,Outbound,Luggage,Sales,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7571315193780600815,M-15958320,be6d487c-7d55-40f3-9031-5ec9d0899c5e,7a6f31db-5f40-530e-928a-e2da6e604b7a,2026-04-19 10:28:50.166000+00:00,CLUG,Email,Outbound,Luggage,Sales,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7571315193780600815,M-15958320,114646cd-7950-41be-94f2-21ef31855029,ef617a11-cc81-5fff-babb-d70f9f6d8b30,2026-04-19 10:28:50.166000+00:00,L5B15,Email,Outbound,Luggage,Sales,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Partition & model version overview

In [5]:
# Product variant breakdown
print("Decisions per luggage variant:")
print(df["pyName"].value_counts(dropna=False).to_string())

print()
dt = df["pxDecisionTime"].dropna()
if len(dt):
    print(f"Interaction date range: {dt.min().date()} → {dt.max().date()}")

Decisions per luggage variant:
pyName
CLUG     2684
L5B15    2268
L5B25     508
L5B30     294
L5B40     176
L5B20     152
L5B50      82

Interaction date range: 2026-03-09 → 2026-04-20


In [6]:
# Propensity by variant
print("Propensity summary by pyName:")
print(df.groupby("pyName")["propensity"].describe().round(4).to_string())

Propensity summary by pyName:
         count    mean     std     min     25%     50%     75%     max
pyName                                                                
CLUG    2684.0  0.0129  0.0139  0.0000  0.0076  0.0115  0.0147  0.4647
L5B15   2268.0  0.0056  0.0063  0.0015  0.0026  0.0042  0.0059  0.0749
L5B20    152.0  0.0094  0.0081  0.0013  0.0049  0.0054  0.0124  0.0451
L5B25    508.0  0.0087  0.0103  0.0032  0.0049  0.0059  0.0067  0.1246
L5B30    294.0  0.0081  0.0068  0.0001  0.0035  0.0075  0.0104  0.0372
L5B40    176.0  0.0113  0.0112  0.0005  0.0056  0.0081  0.0130  0.0923
L5B50     82.0  0.0136  0.0171  0.0031  0.0037  0.0067  0.0128  0.0708


## Save to Parquet

In [7]:
df.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved {len(df):,} rows → {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")

Saved 6,164 rows → ..\data\processed\luggage_email_outbound.parquet
File size: 1708.3 KB
